In [19]:

import os
import openai
from openai import OpenAI

# Diese beiden Codes sind individuell und müssen selbst auf der Website der KI Toolbox generiert werden.
# Leider steht diese Anwendung vorerst nur für Mitarbeiter:innen zur Verfügung.

try:
    api_key = os.environ["KIT_AI_API_KEY"]
except KeyError as exc:
    raise RuntimeError("Umgebungs‑Variable KIT_AI_API_KEY fehlt") from exc

base_url = "https://ki-toolbox.scc.kit.edu/api/v1"


In [20]:
# Initialize Client
client = OpenAI(api_key=api_key, base_url=base_url)

try:
    # Send chat request
    chat_completion = client.chat.completions.create(
    #model='qwen3-vl:235b-a22b',
    #model="azure.gpt-4.1-mini", # Select the appropriate model here
    model='kit.gpt-oss-120b',    
    messages=[
        {"role": "system", "content": "You are a helpful assistant at KIT."},
        {"role": "user", "content": "Wer oder was bist du?."}
    ])
except KeyError as exc:
     raise RuntimeError("Error") from exc


# Output answer
print(chat_completion.choices[0].message.content)

Ich bin ein KI‑gestützter Chatbot – genauer gesagt das Sprachmodell **ChatGPT**, das von OpenAI entwickelt wurde. Hier bei der KIT stehe ich dir als virtueller Assistent zur Verfügung, um Fragen zu beantworten, bei Problemen zu helfen und dich bei allen möglichen Themen zu unterstützen. Wenn du also etwas wissen möchtest oder Unterstützung brauchst, sag einfach Bescheid!


In [23]:
import json

SARCASM_SCHEMA = {
    "sarcasm": {
        "sarcasm_likert": "1-7 integer (1 = not sarcastic, 7 = very sarcastic)",
        "sarcasm_score": "0.0-1.0 float (0 = not sarcastic, 1 = very sarcastic)",
        "justification": "string - explanation based on observed language",
        "sarcasm_markers": ["string"]
    } 
}


def prompt_sarcasm_analysis(comment: str) -> str:
    schema_str = json.dumps(SARCASM_SCHEMA, indent=2)
    return f"""
You are a rigorous JSON-only annotator of online sarcasm.
Analyze the COMMENT below.

Return ONLY one JSON object that strictly matches this schema:

{schema_str}

GUIDELINES:
- Base judgments on lexical and pragmatic cues inside the COMMENT only.
- Quote verbatim evidence for any *present = true* marker (≤ 20 words).
- sarcasm_likert: 1 (highly uncivil) … 7 (very civil). Use 4 for neutral/mixed.
- sarcasm_score: map roughly from Likert to [0,1] (1→0.0, 4→0.5, 7→1.0).
- Output valid JSON only — no explanations.

COMMENT:
\"\"\"{comment}\"\"\"

"""


comment = "Das ist aber toll!"
prompt = prompt_sarcasm_analysis(comment)

print(prompt)









You are a rigorous JSON-only annotator of online sarcasm.
Analyze the COMMENT below.

Return ONLY one JSON object that strictly matches this schema:

{
  "sarcasm": {
    "sarcasm_likert": "1-7 integer (1 = not sarcastic, 7 = very sarcastic)",
    "sarcasm_score": "0.0-1.0 float (0 = not sarcastic, 1 = very sarcastic)",
    "justification": "string - explanation based on observed language",
    "sarcasm_markers": [
      "string"
    ]
  }
}

GUIDELINES:
- Base judgments on lexical and pragmatic cues inside the COMMENT only.
- Quote verbatim evidence for any *present = true* marker (≤ 20 words).
- sarcasm_likert: 1 (highly uncivil) … 7 (very civil). Use 4 for neutral/mixed.
- sarcasm_score: map roughly from Likert to [0,1] (1→0.0, 4→0.5, 7→1.0).
- Output valid JSON only — no explanations.

COMMENT:
"""Das ist aber toll!"""


In [24]:

client = OpenAI(api_key=api_key, base_url=base_url)

try:
    # Send chat request
    chat_completion = client.chat.completions.create(
        #model='qwen3-vl:235b-a22b',
        #model="azure.gpt-4.1-mini", # Select the appropriate model here
        model='kit.gpt-oss-120b',
        messages=[
            {"role": "system", "content": "You are a rigorous JSON-only annotator of online sarcasm."},
            {"role": "user", "content": prompt}
        ])
except KeyError as exc:
    raise RuntimeError("Error") from exc


# Output answer
print(chat_completion.choices[0].message.content)



{
  "sarcasm": {
    "sarcasm_likert": 4,
    "sarcasm_score": 0.5,
    "justification": "The comment \"Das ist aber toll!\" uses \"aber\" as an intensifier that can convey sarcasm, but without surrounding context it is ambiguous, so a neutral rating is appropriate.",
    "sarcasm_markers": [
      "aber"
    ]
  }
}
